In [0]:
from pyspark.sql import functions as F

silver_df = (spark.table("workspace.sales_project.bronze_sales")
    # 1. Standardize column names: lowercase, underscores
    .toDF(*[c.strip().lower().replace(" ", "_").replace("-", "_") for c in
            spark.table("workspace.sales_project.bronze_sales").columns])
    # 2. Remove duplicates
    .dropDuplicates()
    # 3. Fix data types (adjust date format to match your file)
    .withColumn("order_date", F.to_date("order_date", "M/d/yyyy"))
    .withColumn("ship_date", F.to_date("ship_date", "M/d/yyyy"))
    # 4. Drop rows missing critical fields
    .dropna(subset=["order_id", "order_date", "sales"])
    # 5. Filter out bad values
    .filter(F.expr("try_cast(sales AS double)") > 0)
    # 6. Trim text columns
    .withColumn("region", F.trim("region"))
    .withColumn("category", F.trim("category"))
    # 7. Add useful derived columns
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
)

silver_df.write.mode("overwrite").saveAsTable("workspace.sales_project.silver_sales")